# Data Abstraction with AbstractionReviewer

This tutorial demonstrates how to use `AbstractionReviewer` to extract structured data from research article abstracts. The `AbstractionReviewer` dynamically builds a Pydantic output model from the fields you define, enabling flexible and type-safe data extraction.

We will:
1. Load a dataset of 20 agent-based modeling (ABM) research articles
2. Define an abstraction schema with 5 extraction fields
3. Extract structured data from articles using `review_items()`
4. Convert results to a DataFrame for analysis
5. Explore the dynamic output model

**Requirements:** `OPENAI_API_KEY` environment variable must be set.

## Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from pathlib import Path
from lattereview.agentic import AbstractionReviewer

## Load the Dataset

The dataset contains 20 articles about agent-based modeling (ABM) with columns: `Title`, `Abstract`, `Authors`, `Year`.

In [2]:
df = pd.read_csv("data.csv")
print(f"Dataset: {len(df)} articles")
print(f"Columns: {list(df.columns)}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print()
df[["Title", "Year"]].head(5)

Dataset: 20 articles
Columns: ['Title', 'Abstract', 'Authors', 'Year']
Year range: 2017 - 2024



,Title,Year
0,Fusing an agent-based model of mosquito popula...,2022
1,PDRL: Multi-Agent based Reinforcement Learning...,2023
2,Learning-accelerated Discovery of Immune-Tumou...,2019
3,Investigating spatiotemporal dynamics and sync...,2018
4,Modeling the Spread of COVID-19 in University ...,2024


## Define the Abstraction Schema

We define 5 fields to extract from each abstract. The `abstraction_keys` dict maps field names to their Python types, while `key_descriptions` provides detailed extraction guidance for each field.

The reviewer is given the `searching-duckduckgo` skill so it can look up additional context about methodologies or disease areas mentioned in the abstracts.

In [3]:
abstractor = AbstractionReviewer(
    name="ABMExtractor",
    backstory=(
        "You are an expert in agent-based modeling and computational biology. "
        "You extract structured information from research articles about ABM "
        "with precision, identifying the type of study, methodology, and key findings."
    ),
    model="openai:gpt-5.4-mini",
    abstraction_keys={
        "study_type": str,
        "sample_size": str,
        "disease_focus": str,
        "abm_methodology": str,
        "key_finding": str,
    },
    key_descriptions={
        "study_type": (
            "The type of study: e.g., 'simulation study', 'empirical validation', "
            "'methodological development', 'hybrid model', 'review'. "
            "If unclear, describe the study's primary approach."
        ),
        "sample_size": (
            "The number of agents, simulations, patients, or data points used. "
            "Report as a string (e.g., '1000 agents', '500 simulations', 'N/A'). "
            "Use 'not reported' if the abstract does not mention a sample size."
        ),
        "disease_focus": (
            "The disease, condition, or biological system being modeled. "
            "Examples: 'dengue transmission', 'tumor growth', 'COVID-19 spread'. "
            "Use 'general/no specific disease' if not disease-focused."
        ),
        "abm_methodology": (
            "The specific ABM approach or framework used. Include details about "
            "the agent types, interaction rules, or computational framework "
            "(e.g., 'NetLogo', 'MASON', 'custom Python ABM', 'reinforcement learning ABM')."
        ),
        "key_finding": (
            "The primary result or conclusion in one sentence. "
            "Include quantitative metrics if available."
        ),
    },
    max_iterations=10,
    skills=["searching-duckduckgo"],
)

print(f"Reviewer: {abstractor.name}")
print(f"Fields to extract: {list(abstractor.abstraction_keys.keys())}")
print(f"Skills: {abstractor.skills}")

Reviewer: ABMExtractor
Fields to extract: ['study_type', 'sample_size', 'disease_focus', 'abm_methodology', 'key_finding']
Skills: ['searching-duckduckgo']


## Extract Data from Articles

We combine the `Title` and `Abstract` columns into a single text input and run the abstractor on the first 5 articles using `review_items()`.

In [4]:
# Combine title and abstract for richer context
articles = [
    f"Title: {row['Title']}\n\nAbstract: {row['Abstract']}"
    for _, row in df.head(5).iterrows()
]

print(f"Processing {len(articles)} articles...")
print(f"\nExample input (truncated):\n{articles[0][:200]}...")

Processing 5 articles...

Example input (truncated):
Title: Fusing an agent-based model of mosquito population dynamics with a statistical reconstruction of spatio-temporal abundance patterns

Abstract: The mosquito Aedes aegypti is the vector of a numb...


In [5]:
results, total_cost = await abstractor.review_items(articles)

print(f"Extracted data from {len(results)} articles")
print(f"Total cost: ${total_cost:.4f}")
print()

# Display each result
for i, result in enumerate(results):
    print(f"\n--- Article {i + 1}: {df.iloc[i]['Title'][:60]}... ---")
    for key, value in result.items():
        if not key.startswith("_"):
            print(f"  {key}: {value}")

Extracted data from 5 articles
Total cost: $0.0000


--- Article 1: Fusing an agent-based model of mosquito population dynamics ... ---
  study_type: hybrid model / methodological development with empirical validation
  sample_size: 176,352 household-level Ae. aegypti aspirator collections
  disease_focus: Ae. aegypti mosquito dynamics relevant to dengue, yellow fever, chikungunya, and Zika transmission
  abm_methodology: Agent-based model of Ae. aegypti population dynamics calibrated to spatio-temporal abundance patterns from a generalized additive model (GAM), using literature-derived parameters and a single fitted parameter to capture residual variation; model used to simulate adult mosquito abundance and insecticide spraying effects
  key_finding: The calibrated ABM closely matched GAM-predicted baseline mosquito abundance and predicted that mosquito abundance rebounds within about two months after adulticide spraying, consistent with experimental data from Iquitos.

--- Article 2:

## Convert to DataFrame

The results can be easily converted to a pandas DataFrame for further analysis, filtering, or export.

In [6]:
# Build a DataFrame from results
extracted_df = pd.DataFrame(results)

# Drop internal columns (prefixed with _)
display_cols = [c for c in extracted_df.columns if not c.startswith("_")]
extracted_df = extracted_df[display_cols]

# Add article titles for context
extracted_df.insert(0, "title", df["Title"].head(5).values)

print(f"Extracted DataFrame: {extracted_df.shape}")
extracted_df

Extracted DataFrame: (5, 6)


,title,study_type,sample_size,disease_focus,abm_methodology,key_finding
0,Fusing an agent-based model of mosquito popula...,hybrid model / methodological development with...,"176,352 household-level Ae. aegypti aspirator ...",Ae. aegypti mosquito dynamics relevant to deng...,Agent-based model of Ae. aegypti population dy...,The calibrated ABM closely matched GAM-predict...
1,PDRL: Multi-Agent based Reinforcement Learning...,methodological development / simulation study,3 DRL agents; no patient or dataset size reported,general/no specific disease,predictive deep reinforcement learning (PDRL) ...,The proposed PDRL framework learned future-sta...
2,Learning-accelerated Discovery of Immune-Tumou...,methodological development / simulation study,not reported,cancer immunotherapy / heterogeneous tumour im...,PhysiCell agent-based model of immunosurveilla...,The integrated ABM-HPC framework enabled adapt...
3,Investigating spatiotemporal dynamics and sync...,simulation study,19.8 million stochastically generated software...,influenza epidemics in Australia,"AceMod, a custom agent-based modelling framewo...",The model synthesized influenza epidemics acro...
4,Modeling the Spread of COVID-19 in University ...,simulation study,"14,000 students and faculty (synthetic univers...",COVID-19 spread,Stochastic agent-based SEIR model of a univers...,The ABM produced similar results to the determ...


## The Dynamic Output Model

`AbstractionReviewer` uses `build_dynamic_output_model()` to construct a Pydantic model at runtime from your `abstraction_keys`. This means you can define any combination of `str`, `int`, `float`, `bool`, or `list` types and the reviewer will validate the LLM output against this schema.

Let's inspect the generated model.

In [7]:
# The output_type is a dynamically-built Pydantic model
output_model = abstractor.output_type
print(f"Model name: {output_model.__name__}")
print(f"\nFields:")
for field_name, field_info in output_model.model_fields.items():
    print(f"  {field_name}: {field_info.annotation.__name__}")

print(f"\nJSON Schema:")
import json
print(json.dumps(output_model.model_json_schema(), indent=2))

Model name: AbstractionResult

Fields:
  study_type: str
  sample_size: str
  disease_focus: str
  abm_methodology: str
  key_finding: str

JSON Schema:
{
  "properties": {
    "study_type": {
      "title": "Study Type",
      "type": "string"
    },
    "sample_size": {
      "title": "Sample Size",
      "type": "string"
    },
    "disease_focus": {
      "title": "Disease Focus",
      "type": "string"
    },
    "abm_methodology": {
      "title": "Abm Methodology",
      "type": "string"
    },
    "key_finding": {
      "title": "Key Finding",
      "type": "string"
    }
  },
  "required": [
    "study_type",
    "sample_size",
    "disease_focus",
    "abm_methodology",
    "key_finding"
  ],
  "title": "AbstractionResult",
  "type": "object"
}


## Clean Up

In [8]:
import shutil

# Clean up any working directories created during the review
for d in Path(".").glob("agent_*"):
    if d.is_dir():
        shutil.rmtree(d)
        print(f"Removed {d}")

print("Done.")

Done.
